# Advanced `MappingProxyType`: Read-Only Live Views of Mappings

This notebook develops a precise mental model for `types.MappingProxyType`, then applies it through progressively harder examples and solved problems.

## Learning goals

By the end, you should be able to:

- distinguish a **read-only live view** from a copied snapshot;
- explain why a mapping proxy is only **shallowly** read-only;
- design APIs that expose state without exposing direct mutation;
- use `Mapping` instead of `dict` in type hints when callers only need read access;
- reason about class namespaces such as `SomeClass.__dict__`;
- test the guarantees that a proxy does and does not provide;
- recognize concurrency, aliasing, and nested-mutability pitfalls;
- build a small configuration/feature-flag registry with both live views and snapshots.

> Best-practice vocabulary: a `mappingproxy` is not a frozen dictionary. It prevents mutation **through the proxy**, while reflecting changes made through the wrapped mapping.

## 1. Setup

In [1]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass
from types import MappingProxyType
from typing import Any, Mapping, MutableMapping

## 2. Core behavior: proxy versus source mapping

`MappingProxyType(source)` creates a read-only wrapper around `source`. Reads are delegated to the current state of the wrapped mapping.

In [2]:
source = {"host": "localhost", "port": 8000}
view = MappingProxyType(source)

print(type(source))
print(type(view))
print(view["host"])
print(dict(view))

<class 'dict'>
<class 'mappingproxy'>
localhost
{'host': 'localhost', 'port': 8000}


The proxy supports normal read operations such as indexing, membership tests, iteration, `keys()`, `values()`, `items()`, `get()`, and `len()`.

In [3]:
assert view["port"] == 8000
assert "host" in view
assert len(view) == 2
assert view.get("missing", "fallback") == "fallback"

print(list(view))
print(list(view.keys()))
print(list(view.values()))
print(list(view.items()))

['host', 'port']
['host', 'port']
['localhost', 8000]
[('host', 'localhost'), ('port', 8000)]


## 3. Mutation through the proxy is blocked

In [4]:
operations = [
    ("item assignment", lambda: view.__setitem__("port", 9000)),
    ("item deletion", lambda: view.__delitem__("host")),
]

for label, operation in operations:
    try:
        operation()
    except (AttributeError, TypeError) as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")

item assignment: AttributeError: 'mappingproxy' object has no attribute '__setitem__'
item deletion: AttributeError: 'mappingproxy' object has no attribute '__delitem__'


Direct syntax such as `view["port"] = 9000` is also rejected. The special-method calls above are used only so several failure cases can be demonstrated in one loop.

In [5]:
try:
    view["port"] = 9000
except TypeError as exc:
    print(type(exc).__name__, exc)

TypeError 'mappingproxy' object does not support item assignment


## 4. The proxy is live, not a snapshot

Mutating the original mapping changes what the proxy sees.

In [6]:
source["port"] = 9001
source["debug"] = True
del source["host"]

print(source)
print(view)

assert view["port"] == 9001
assert view["debug"] is True
assert "host" not in view

{'port': 9001, 'debug': True}
{'port': 9001, 'debug': True}


## 5. Snapshot versus live view

A copy and a proxy solve different problems:

- `MappingProxyType(source)` gives a **read-only live view**.
- `dict(source)` or `view.copy()` gives a **mutable shallow snapshot**.

In [7]:
source = {"a": 1, "b": 2}
live = MappingProxyType(source)
snapshot = live.copy()

source["a"] = 999
source["c"] = 3

print("source:  ", source)
print("live:    ", live)
print("snapshot:", snapshot)

assert live["a"] == 999
assert snapshot["a"] == 1
assert "c" in live
assert "c" not in snapshot
assert isinstance(snapshot, dict)

source:   {'a': 999, 'b': 2, 'c': 3}
live:     {'a': 999, 'b': 2, 'c': 3}
snapshot: {'a': 1, 'b': 2}


## 6. Important pitfall: read-only is shallow

A proxy blocks replacing or deleting entries through the proxy, but it does **not** recursively freeze mutable objects stored as values.

In [8]:
source = {
    "service": "analytics",
    "tags": ["stable", "internal"],
    "limits": {"requests": 100},
}
view = MappingProxyType(source)

# These mutate nested objects, not the proxy's key/value structure.
view["tags"].append("priority")
view["limits"]["requests"] = 250

print(view)

{'service': 'analytics', 'tags': ['stable', 'internal', 'priority'], 'limits': {'requests': 250}}


If deep immutability is required, `MappingProxyType` by itself is insufficient. One option is to copy and recursively convert mutable containers to immutable representations.

In [9]:
def deep_freeze(value: Any) -> Any:
    """Recursively convert common mutable containers to immutable equivalents."""
    if isinstance(value, dict):
        frozen_dict = {k: deep_freeze(v) for k, v in value.items()}
        return MappingProxyType(frozen_dict)
    if isinstance(value, list):
        return tuple(deep_freeze(v) for v in value)
    if isinstance(value, set):
        return frozenset(deep_freeze(v) for v in value)
    if isinstance(value, tuple):
        return tuple(deep_freeze(v) for v in value)
    return value


config = {
    "database": {"hosts": ["db1", "db2"]},
    "roles": {"reader", "writer"},
}
frozen_config = deep_freeze(config)

print(frozen_config)
print(frozen_config["database"]["hosts"])
print(frozen_config["roles"])

{'database': mappingproxy({'hosts': ('db1', 'db2')}), 'roles': frozenset({'writer', 'reader'})}
('db1', 'db2')
frozenset({'writer', 'reader'})


`deep_freeze` is an application-level convention, not a universal serializer. Custom mutable objects may still need custom handling.

# Advanced solved problems

## Problem 1 — Prove the exact guarantee

Create a source dictionary and a proxy. Demonstrate with assertions that:

1. reads through the proxy work;
2. writes through the proxy fail;
3. writes through the source remain visible through the proxy.

### Solution

In [10]:
data = {"x": 10}
proxy = MappingProxyType(data)

# 1. Reads work.
assert proxy["x"] == 10

# 2. Writes through the proxy fail.
try:
    proxy["x"] = 20
except TypeError:
    pass
else:
    raise AssertionError("Expected assignment through the proxy to fail")

# 3. Source mutations are visible through the proxy.
data["x"] = 20
data["y"] = 30

assert proxy["x"] == 20
assert proxy["y"] == 30

print("All guarantees demonstrated successfully.")

All guarantees demonstrated successfully.


## Problem 2 — Build a safe public configuration API

Design a class that owns a mutable internal dictionary but exposes a read-only live view to consumers.

Requirements:

- callers may read configuration;
- callers may not assign through the returned object;
- the owner can update values through a method;
- previously obtained views must reflect later owner updates.

### Solution

In [11]:
class Configuration:
    def __init__(self, initial: Mapping[str, Any] | None = None) -> None:
        self._data: dict[str, Any] = dict(initial or {})
        self._view: Mapping[str, Any] = MappingProxyType(self._data)

    @property
    def values(self) -> Mapping[str, Any]:
        return self._view

    def set(self, key: str, value: Any) -> None:
        self._data[key] = value

    def remove(self, key: str) -> None:
        del self._data[key]


config = Configuration({"timeout": 10})
public_view = config.values

assert public_view["timeout"] == 10

config.set("timeout", 30)
config.set("retries", 5)

assert public_view["timeout"] == 30
assert public_view["retries"] == 5

try:
    public_view["timeout"] = 999
except TypeError as exc:
    print("Blocked external mutation:", exc)

print(public_view)

Blocked external mutation: 'mappingproxy' object does not support item assignment
{'timeout': 30, 'retries': 5}


### Why this design is useful

The owner preserves a controlled mutation path (`set`, `remove`) while consumers receive the narrower `Mapping` interface. This is often clearer than returning a writable dictionary and merely documenting "please do not modify it."

## Problem 3 — Find the aliasing bug

The following design appears read-only, but callers can still mutate nested state:

```python
settings = {"plugins": ["auth", "metrics"]}
public = MappingProxyType(settings)
```

Show the bug, then produce a safer snapshot for untrusted consumers.

### Solution

In [12]:
settings = {"plugins": ["auth", "metrics"]}
public = MappingProxyType(settings)

# The list itself is still mutable.
public["plugins"].append("debug")
assert settings["plugins"] == ["auth", "metrics", "debug"]

# A defensive deep copy breaks aliases to nested containers.
safe_snapshot = deepcopy(dict(public))

safe_snapshot["plugins"].append("local-only")

print("internal:", settings)
print("snapshot:", safe_snapshot)

assert "local-only" not in settings["plugins"]

internal: {'plugins': ['auth', 'metrics', 'debug']}
snapshot: {'plugins': ['auth', 'metrics', 'debug', 'local-only']}


A deep copy provides separation, not read-only behavior. If the consumer must also receive an immutable representation, combine defensive copying with an appropriate freezing strategy.

## Problem 4 — Return a snapshot or a live view?

Implement both behaviors in one registry:

- `view()` returns a read-only live view;
- `snapshot()` returns an independent dictionary representing the current state.

### Solution

In [13]:
class Registry:
    def __init__(self) -> None:
        self._items: dict[str, Any] = {}
        self._view = MappingProxyType(self._items)

    def register(self, name: str, value: Any) -> None:
        self._items[name] = value

    def view(self) -> Mapping[str, Any]:
        return self._view

    def snapshot(self) -> dict[str, Any]:
        return self._items.copy()


registry = Registry()
registry.register("alpha", 1)

live = registry.view()
snap = registry.snapshot()

registry.register("beta", 2)

print("live:", live)
print("snap:", snap)

assert "beta" in live
assert "beta" not in snap

live: {'alpha': 1, 'beta': 2}
snap: {'alpha': 1}


## Problem 5 — Use the correct type hint

A function only reads from its input mapping. Refactor it so that it accepts dictionaries, mapping proxies, and other mapping implementations without falsely requiring mutability.

### Solution

In [14]:
def connection_string(config: Mapping[str, Any]) -> str:
    host = config.get("host", "localhost")
    port = config.get("port", 5432)
    return f"{host}:{port}"


plain_dict = {"host": "db.example.com", "port": 6543}
read_only = MappingProxyType(plain_dict)

assert connection_string(plain_dict) == "db.example.com:6543"
assert connection_string(read_only) == "db.example.com:6543"

print(connection_string(read_only))

db.example.com:6543


**Best practice:** annotate read-only parameters with `Mapping[K, V]`. Use `MutableMapping[K, V]` only when the function actually mutates the mapping.

## Problem 6 — Implement controlled bulk updates

Extend the configuration API with an `update()` method while keeping the public view read-only.

Requirements:

- update multiple keys at once;
- optionally reject unknown keys;
- keep one stable proxy object instead of constructing a new proxy for every read.

### Solution

In [15]:
class StrictConfiguration:
    def __init__(self, initial: Mapping[str, Any]) -> None:
        self._data = dict(initial)
        self._view = MappingProxyType(self._data)

    @property
    def values(self) -> Mapping[str, Any]:
        return self._view

    def update(
        self,
        changes: Mapping[str, Any],
        *,
        allow_new: bool = False,
    ) -> None:
        if not allow_new:
            unknown = changes.keys() - self._data.keys()
            if unknown:
                raise KeyError(f"Unknown configuration keys: {sorted(unknown)}")
        self._data.update(changes)


cfg = StrictConfiguration({"timeout": 10, "retries": 3})
view_before = cfg.values

cfg.update({"timeout": 20})

assert cfg.values is view_before
assert view_before["timeout"] == 20

try:
    cfg.update({"region": "eu"})
except KeyError as exc:
    print(exc)

cfg.update({"region": "eu"}, allow_new=True)
assert view_before["region"] == "eu"

print(view_before)

"Unknown configuration keys: ['region']"
{'timeout': 20, 'retries': 3, 'region': 'eu'}


## Problem 7 — Validate before mutating shared state

Modify a registry so failed updates do not leave partially applied state.

The update should:

- accept a mapping of changes;
- validate all keys and values first;
- mutate the internal dictionary only after validation succeeds.

### Solution

In [16]:
class PortConfiguration:
    def __init__(self) -> None:
        self._data = {"http": 80, "https": 443}
        self.view = MappingProxyType(self._data)

    @staticmethod
    def _validate_port(name: str, value: Any) -> None:
        if not isinstance(value, int):
            raise TypeError(f"{name!r} must be an integer")
        if not 1 <= value <= 65535:
            raise ValueError(f"{name!r} must be in 1..65535")

    def update(self, changes: Mapping[str, Any]) -> None:
        # Validate every proposed change first.
        for name, value in changes.items():
            self._validate_port(name, value)

        # Only now mutate shared state.
        self._data.update(changes)


ports = PortConfiguration()

try:
    ports.update({"admin": 9000, "broken": 99999})
except ValueError as exc:
    print("Rejected:", exc)

# No partial update occurred.
assert "admin" not in ports.view
assert "broken" not in ports.view

ports.update({"admin": 9000})
assert ports.view["admin"] == 9000

print(ports.view)

Rejected: 'broken' must be in 1..65535
{'http': 80, 'https': 443, 'admin': 9000}


## Problem 8 — Understand class namespaces

Python exposes a class namespace through a mapping proxy. Inspect it, show that direct item assignment fails, then modify the class through the supported attribute API.

### Solution

In [17]:
class Service:
    protocol = "https"

    def endpoint(self) -> str:
        return "/health"


namespace = Service.__dict__

print(type(namespace))
print(namespace["protocol"])
print("endpoint" in namespace)

try:
    namespace["protocol"] = "http"
except TypeError as exc:
    print("Direct namespace mutation blocked:", exc)

# Supported class mutation goes through attribute operations.
setattr(Service, "protocol", "http")
setattr(Service, "version", 2)

assert namespace["protocol"] == "http"
assert namespace["version"] == 2

print(Service.__dict__)

<class 'mappingproxy'>
https
True
Direct namespace mutation blocked: 'mappingproxy' object does not support item assignment
{'__module__': '__main__', '__firstlineno__': 1, 'protocol': 'http', 'endpoint': <function Service.endpoint at 0x00000210C6DCC4A0>, '__static_attributes__': (), '__dict__': <attribute '__dict__' of 'Service' objects>, '__weakref__': <attribute '__weakref__' of 'Service' objects>, '__doc__': None, 'version': 2}


This illustrates the same core rule: the proxy blocks direct mapping mutation while still reflecting legitimate changes to the underlying class namespace.

## Problem 9 — Write a function that accepts either a proxy or dictionary

Implement `select_keys(mapping, keys)` and return a plain dictionary containing only requested keys that exist.

### Solution

In [18]:
def select_keys(
    mapping: Mapping[str, Any],
    keys: list[str],
) -> dict[str, Any]:
    return {key: mapping[key] for key in keys if key in mapping}


base = {"a": 1, "b": 2, "c": 3}
proxy = MappingProxyType(base)

assert select_keys(base, ["a", "c", "z"]) == {"a": 1, "c": 3}
assert select_keys(proxy, ["b", "z"]) == {"b": 2}

print(select_keys(proxy, ["a", "b"]))

{'a': 1, 'b': 2}


## Problem 10 — Preserve encapsulation when values are mutable

Create a settings service whose public state must not expose mutable nested containers owned by the service.

Use an immutable nested representation for a list of servers.

### Solution

In [19]:
class ServerSettings:
    def __init__(self, servers: list[str]) -> None:
        # Store an immutable tuple internally.
        self._data: dict[str, Any] = {
            "servers": tuple(servers),
            "strategy": "round-robin",
        }
        self._view = MappingProxyType(self._data)

    @property
    def values(self) -> Mapping[str, Any]:
        return self._view

    def replace_servers(self, servers: list[str]) -> None:
        # Replace the tuple via the owned mutable dictionary.
        self._data["servers"] = tuple(servers)


settings = ServerSettings(["s1", "s2"])
public = settings.values

assert public["servers"] == ("s1", "s2")

settings.replace_servers(["s3", "s4"])
assert public["servers"] == ("s3", "s4")

print(public)

{'servers': ('s3', 's4'), 'strategy': 'round-robin'}


This pattern is stronger than merely wrapping a dictionary: the outer mapping cannot be mutated through the proxy, and the exposed server collection is itself immutable.

## Problem 11 — Compare identity and equality

Determine what changes when the source dictionary is updated:

- identity of the proxy;
- equality of the proxy to a regular dictionary;
- contents visible through the proxy.

### Solution

In [20]:
data = {"count": 1}
proxy = MappingProxyType(data)
same_proxy = proxy

assert proxy is same_proxy
assert proxy == {"count": 1}

data["count"] += 1

assert proxy is same_proxy
assert proxy == {"count": 2}
assert dict(proxy) == {"count": 2}

print("id(proxy):", id(proxy))
print("contents:", dict(proxy))

id(proxy): 2271078752768
contents: {'count': 2}


## Problem 12 — Avoid accidentally leaking the mutable source

Identify the flaw in the first class and fix it.

### Buggy version

In [21]:
class BuggyStore:
    def __init__(self) -> None:
        self._data = {"secret_mode": False}

    @property
    def data(self) -> dict[str, Any]:
        # BUG: callers receive the actual mutable dictionary.
        return self._data


buggy = BuggyStore()
leaked = buggy.data
leaked["secret_mode"] = True

assert buggy.data["secret_mode"] is True
print("Internal state was mutated externally:", buggy.data)

Internal state was mutated externally: {'secret_mode': True}


### Corrected version

In [22]:
class SaferStore:
    def __init__(self) -> None:
        self._data = {"secret_mode": False}
        self._view = MappingProxyType(self._data)

    @property
    def data(self) -> Mapping[str, Any]:
        return self._view

    def set_secret_mode(self, enabled: bool) -> None:
        self._data["secret_mode"] = bool(enabled)


store = SaferStore()
public = store.data

try:
    public["secret_mode"] = True
except TypeError:
    pass

assert store.data["secret_mode"] is False

store.set_secret_mode(True)
assert public["secret_mode"] is True

print(public)

{'secret_mode': True}


## Problem 13 — Read-only does not mean thread-safe

Explain why a mapping proxy does not automatically make shared mutable state safe for concurrent mutation, then implement a simple lock-protected registry.

### Solution

A proxy only controls *which interface can perform writes*. If another thread can mutate the wrapped dictionary, readers may observe state changing between operations. Multi-step invariants still require synchronization or another concurrency strategy.

In [23]:
from threading import RLock


class LockedRegistry:
    def __init__(self) -> None:
        self._data: dict[str, int] = {}
        self._view = MappingProxyType(self._data)
        self._lock = RLock()

    @property
    def view(self) -> Mapping[str, int]:
        # Individual dictionary reads are exposed through the proxy.
        # Cross-key invariants still need lock-aware methods.
        return self._view

    def set(self, key: str, value: int) -> None:
        with self._lock:
            self._data[key] = value

    def snapshot(self) -> dict[str, int]:
        # A consistent copy is produced while holding the lock.
        with self._lock:
            return self._data.copy()


locked = LockedRegistry()
locked.set("jobs", 4)
locked.set("workers", 2)

print("live view:", locked.view)
print("locked snapshot:", locked.snapshot())

live view: {'jobs': 4, 'workers': 2}
locked snapshot: {'jobs': 4, 'workers': 2}


## Problem 14 — Build a read-only computed index

Given a sequence of records, build an index by ID and expose that index through a mapping proxy. Then replace the entire index in a controlled refresh operation while preserving a stable public proxy.

### Solution

In [24]:
@dataclass(frozen=True)
class User:
    user_id: int
    name: str


class UserIndex:
    def __init__(self, users: list[User]) -> None:
        self._index: dict[int, User] = {}
        self._view = MappingProxyType(self._index)
        self.refresh(users)

    @property
    def by_id(self) -> Mapping[int, User]:
        return self._view

    def refresh(self, users: list[User]) -> None:
        rebuilt = {user.user_id: user for user in users}

        if len(rebuilt) != len(users):
            raise ValueError("Duplicate user_id detected")

        # Keep the same dict object so existing proxies remain live.
        self._index.clear()
        self._index.update(rebuilt)


index = UserIndex([
    User(1, "Ada"),
    User(2, "Grace"),
])

public_index = index.by_id
assert public_index[1].name == "Ada"

index.refresh([
    User(2, "Grace"),
    User(3, "Linus"),
])

assert public_index[3].name == "Linus"
assert 1 not in public_index

print(public_index)

{2: User(user_id=2, name='Grace'), 3: User(user_id=3, name='Linus')}


### Design lesson

If consumers retain a proxy, replacing `self._index` with a brand-new dictionary would leave old proxies attached to the old object. Mutating the existing dictionary with `clear()` + `update()` preserves the live-view contract.

## Problem 15 — Demonstrate the stale-proxy bug

Write a deliberately incorrect implementation that replaces its internal dictionary. Show why an already-returned proxy becomes stale, then fix the implementation.

### Solution

In [25]:
class StaleProxyRegistry:
    def __init__(self) -> None:
        self._data = {"version": 1}
        self._view = MappingProxyType(self._data)

    @property
    def view(self) -> Mapping[str, int]:
        return self._view

    def bad_refresh(self) -> None:
        # BUG: the proxy still wraps the old dictionary object.
        self._data = {"version": 2}


broken = StaleProxyRegistry()
old_view = broken.view
broken.bad_refresh()

print("new internal dict:", broken._data)
print("old proxy:", old_view)

assert broken._data["version"] == 2
assert old_view["version"] == 1

new internal dict: {'version': 2}
old proxy: {'version': 1}


In [26]:
class StableProxyRegistry:
    def __init__(self) -> None:
        self._data = {"version": 1}
        self._view = MappingProxyType(self._data)

    @property
    def view(self) -> Mapping[str, int]:
        return self._view

    def refresh(self) -> None:
        self._data.clear()
        self._data.update({"version": 2})


fixed = StableProxyRegistry()
stable_view = fixed.view
fixed.refresh()

assert stable_view["version"] == 2
print(stable_view)

{'version': 2}


## Problem 16 — Copy semantics with nested values

Show that `proxy.copy()` is shallow. Then contrast it with `deepcopy(dict(proxy))`.

### Solution

In [27]:
source = {
    "name": "pipeline",
    "steps": ["extract", "transform"],
}

proxy = MappingProxyType(source)
shallow = proxy.copy()
deep = deepcopy(dict(proxy))

source["steps"].append("load")

print("proxy:  ", proxy)
print("shallow:", shallow)
print("deep:   ", deep)

# The shallow copy shares the nested list.
assert shallow["steps"] == ["extract", "transform", "load"]

# The deep copy has an independent nested list.
assert deep["steps"] == ["extract", "transform"]

proxy:   {'name': 'pipeline', 'steps': ['extract', 'transform', 'load']}
shallow: {'name': 'pipeline', 'steps': ['extract', 'transform', 'load']}
deep:    {'name': 'pipeline', 'steps': ['extract', 'transform']}


## Problem 17 — Expose metadata without exposing mutation

Create a decorator that attaches mutable metadata internally but presents it publicly through a mapping proxy.

### Solution

In [28]:
def with_metadata(**initial_metadata: Any):
    def decorator(func):
        metadata = dict(initial_metadata)
        public_metadata = MappingProxyType(metadata)

        func.metadata = public_metadata

        def update_metadata(**changes: Any) -> None:
            metadata.update(changes)

        func.update_metadata = update_metadata
        return func

    return decorator


@with_metadata(category="math", stable=True)
def add(a: int, b: int) -> int:
    return a + b


assert add.metadata["category"] == "math"

add.update_metadata(stable=False, version=2)

assert add.metadata["stable"] is False
assert add.metadata["version"] == 2

try:
    add.metadata["category"] = "other"
except TypeError:
    pass

print(add.metadata)

{'category': 'math', 'stable': False, 'version': 2}


## Problem 18 — Create an immutable top-level default table

Use a module-style mapping proxy for constants so accidental top-level assignment is rejected.

### Solution

In [29]:
_DEFAULTS_DICT = {
    "timeout": 30,
    "retries": 3,
    "backoff": 1.5,
}

DEFAULTS: Mapping[str, float | int] = MappingProxyType(_DEFAULTS_DICT)

assert DEFAULTS["timeout"] == 30

try:
    DEFAULTS["timeout"] = 60
except TypeError as exc:
    print("Protected defaults:", exc)

Protected defaults: 'mappingproxy' object does not support item assignment


For true constants, avoid retaining or exposing `_DEFAULTS_DICT` outside the owning module. Anyone holding the original dictionary can still mutate it.

# Mini-project — Feature flag service with live views and snapshots

## Requirements

Implement a feature-flag service with the following properties:

1. Internal flags are mutable only through service methods.
2. `flags` exposes a read-only live view.
3. `snapshot()` returns an independent dictionary.
4. Flag names must be non-empty strings.
5. Flag values must be booleans.
6. Bulk updates must be atomic with respect to validation: if one change is invalid, no changes are applied.
7. Existing live views must keep working after updates.
8. Include a small test suite using plain `assert` statements.

## Solution

In [30]:
class FeatureFlags:
    def __init__(self, initial: Mapping[str, bool] | None = None) -> None:
        initial_dict = dict(initial or {})
        self._validate_changes(initial_dict)

        self._flags: dict[str, bool] = initial_dict
        self._view: Mapping[str, bool] = MappingProxyType(self._flags)

    @staticmethod
    def _validate_name(name: Any) -> None:
        if not isinstance(name, str):
            raise TypeError("Flag names must be strings")
        if not name.strip():
            raise ValueError("Flag names must not be empty")

    @staticmethod
    def _validate_value(value: Any) -> None:
        if type(value) is not bool:
            raise TypeError("Flag values must be bool")

    @classmethod
    def _validate_changes(cls, changes: Mapping[str, Any]) -> None:
        for name, value in changes.items():
            cls._validate_name(name)
            cls._validate_value(value)

    @property
    def flags(self) -> Mapping[str, bool]:
        return self._view

    def set(self, name: str, enabled: bool) -> None:
        self._validate_name(name)
        self._validate_value(enabled)
        self._flags[name] = enabled

    def update(self, changes: Mapping[str, bool]) -> None:
        # Validate the complete batch before mutating.
        self._validate_changes(changes)
        self._flags.update(changes)

    def remove(self, name: str) -> None:
        del self._flags[name]

    def is_enabled(self, name: str, *, default: bool = False) -> bool:
        return self._flags.get(name, default)

    def snapshot(self) -> dict[str, bool]:
        return self._flags.copy()

In [31]:
# Test 1: initial state
flags = FeatureFlags({
    "new_checkout": False,
    "recommendations": True,
})

live = flags.flags

assert live["new_checkout"] is False
assert live["recommendations"] is True

In [32]:
# Test 2: external writes are blocked
try:
    live["new_checkout"] = True
except TypeError:
    pass
else:
    raise AssertionError("The public mapping must be read-only")

In [33]:
# Test 3: service updates are reflected in an existing live view
flags.set("new_checkout", True)

assert live["new_checkout"] is True
assert flags.is_enabled("new_checkout") is True

In [34]:
# Test 4: bulk update
flags.update({
    "recommendations": False,
    "search_v2": True,
})

assert live["recommendations"] is False
assert live["search_v2"] is True

In [35]:
# Test 5: invalid batch must not partially apply
before = flags.snapshot()

try:
    flags.update({
        "valid_name": True,
        "broken_value": 1,  # int is intentionally rejected
    })
except TypeError as exc:
    print("Rejected invalid batch:", exc)

after = flags.snapshot()

assert after == before
assert "valid_name" not in live

Rejected invalid batch: Flag values must be bool


In [36]:
# Test 6: snapshot is independent at the top level
snapshot = flags.snapshot()
snapshot["local_only"] = True

assert "local_only" not in live

In [37]:
# Test 7: removal remains visible through the same proxy
flags.remove("search_v2")

assert "search_v2" not in live
print("Final live state:", live)

Final live state: {'new_checkout': True, 'recommendations': False}


# Challenge problems

## Challenge 1 — Transaction-style context

Design a method that accepts a proposed mapping of changes, validates everything, and returns a **read-only preview** without mutating live state. Then add a separate `commit()` method.

### One solution

In [38]:
class PreviewableConfig:
    def __init__(self, initial: Mapping[str, int]) -> None:
        self._data = dict(initial)
        self._view = MappingProxyType(self._data)

    @property
    def view(self) -> Mapping[str, int]:
        return self._view

    def preview(self, changes: Mapping[str, int]) -> Mapping[str, int]:
        for key, value in changes.items():
            if not isinstance(key, str):
                raise TypeError("keys must be strings")
            if not isinstance(value, int):
                raise TypeError("values must be integers")

        proposed = self._data | dict(changes)
        return MappingProxyType(proposed)

    def commit(self, changes: Mapping[str, int]) -> None:
        # Reuse preview for validation.
        self.preview(changes)
        self._data.update(changes)


pc = PreviewableConfig({"workers": 2})
proposal = pc.preview({"workers": 8, "queue": 100})

assert proposal["workers"] == 8
assert pc.view["workers"] == 2

pc.commit({"workers": 8, "queue": 100})

assert pc.view["workers"] == 8
assert pc.view["queue"] == 100

## Challenge 2 — Read-only reverse index

Build a structure that maps a group name to a tuple of users and also exposes a reverse mapping from user to group. Both public mappings should be read-only and remain live across rebuilds.

### One solution

In [39]:
class GroupIndex:
    def __init__(self) -> None:
        self._groups: dict[str, tuple[str, ...]] = {}
        self._reverse: dict[str, str] = {}

        self.groups = MappingProxyType(self._groups)
        self.user_to_group = MappingProxyType(self._reverse)

    def rebuild(self, groups: Mapping[str, list[str]]) -> None:
        next_groups: dict[str, tuple[str, ...]] = {}
        next_reverse: dict[str, str] = {}

        for group, users in groups.items():
            immutable_users = tuple(users)
            next_groups[group] = immutable_users

            for user in immutable_users:
                if user in next_reverse:
                    raise ValueError(f"User {user!r} appears in multiple groups")
                next_reverse[user] = group

        # Only mutate live dictionaries after all validation succeeds.
        self._groups.clear()
        self._groups.update(next_groups)

        self._reverse.clear()
        self._reverse.update(next_reverse)


group_index = GroupIndex()
groups_view = group_index.groups
reverse_view = group_index.user_to_group

group_index.rebuild({
    "admin": ["alice", "bob"],
    "viewer": ["carol"],
})

assert groups_view["admin"] == ("alice", "bob")
assert reverse_view["carol"] == "viewer"

print(groups_view)
print(reverse_view)

{'admin': ('alice', 'bob'), 'viewer': ('carol',)}
{'alice': 'admin', 'bob': 'admin', 'carol': 'viewer'}


## Challenge 3 — Recursive read-only conversion

Enhance the earlier `deep_freeze` function so it handles dictionaries, lists, tuples, and sets. Verify that attempts to mutate at several levels fail.

### One solution

In [40]:
nested = {
    "pipeline": {
        "steps": ["extract", "transform", "load"],
        "owners": {"data", "platform"},
    }
}

frozen = deep_freeze(nested)

assert isinstance(frozen, type(MappingProxyType({})))
assert isinstance(frozen["pipeline"], type(MappingProxyType({})))
assert isinstance(frozen["pipeline"]["steps"], tuple)
assert isinstance(frozen["pipeline"]["owners"], frozenset)

failures = 0

try:
    frozen["new"] = 1
except TypeError:
    failures += 1

try:
    frozen["pipeline"]["new"] = 2
except TypeError:
    failures += 1

try:
    frozen["pipeline"]["steps"].append("publish")
except AttributeError:
    failures += 1

try:
    frozen["pipeline"]["owners"].add("security")
except AttributeError:
    failures += 1

assert failures == 4
print("All nested mutation attempts were blocked.")

All nested mutation attempts were blocked.


# Additional short drills with solutions

## Drill A — Membership, iteration, and lookup

In [41]:
d = {"alpha": 10, "beta": 20}
p = MappingProxyType(d)

assert "alpha" in p
assert "gamma" not in p
assert list(p) == ["alpha", "beta"]
assert p.get("beta") == 20
assert p.get("gamma", 0) == 0

## Drill B — Convert a proxy to a regular dictionary

In [42]:
regular = dict(p)

assert type(regular) is dict
regular["alpha"] = 999

assert p["alpha"] == 10
assert regular["alpha"] == 999

## Drill C — A proxy over an empty mapping

In [43]:
empty_source: dict[str, int] = {}
empty_proxy = MappingProxyType(empty_source)

assert len(empty_proxy) == 0

empty_source["first"] = 1

assert len(empty_proxy) == 1
assert empty_proxy["first"] == 1

## Drill D — Deletion from the source

In [44]:
source = {"keep": 1, "remove": 2}
proxy = MappingProxyType(source)

del source["remove"]

assert "remove" not in proxy
assert proxy == {"keep": 1}

## Drill E — Stable proxy identity

In [45]:
class Cache:
    def __init__(self) -> None:
        self._data: dict[str, int] = {}
        self._public = MappingProxyType(self._data)

    @property
    def public(self) -> Mapping[str, int]:
        return self._public

    def put(self, key: str, value: int) -> None:
        self._data[key] = value


cache = Cache()

first = cache.public
second = cache.public

assert first is second

cache.put("hits", 1)
assert first["hits"] == 1

# Common misconceptions

1. **"A mapping proxy makes the original dictionary immutable."**  
   False. The original mapping can still be changed by code that holds it.

2. **"Values inside the proxy are automatically immutable."**  
   False. Nested lists, dictionaries, sets, and custom mutable objects remain mutable unless separately protected.

3. **"A proxy is the same as a copy."**  
   False. A proxy is live; a copy is a snapshot.

4. **"A proxy automatically makes concurrent access safe."**  
   False. Concurrency guarantees require synchronization or an appropriate concurrent design.

5. **"Returning a new proxy on every property access is necessary."**  
   Usually not. If the same source dictionary remains in use, one cached proxy is simpler and preserves identity.

6. **"Replacing the internal dictionary preserves old proxies."**  
   False. Existing proxies keep wrapping the old dictionary object. Mutate the existing dictionary if old proxies must stay live.

# Best-practice checklist

- Prefer `Mapping[K, V]` in public read-only interfaces.
- Keep the mutable backing mapping private.
- Cache one proxy when a stable live view is part of the API.
- Do not call the result "deeply immutable" unless nested values are also protected.
- Use `.copy()` or `dict(proxy)` when callers need an independent top-level snapshot.
- Use `deepcopy` only when its semantics are appropriate for the stored objects.
- Validate a complete batch before mutating shared state.
- If existing proxies must remain live, mutate the backing dictionary instead of rebinding it.
- Do not mistake access control for thread safety.
- Write tests for both positive behavior (reads, live updates) and negative behavior (blocked writes).

# Final review question

For each requirement below, choose the most suitable approach:

| Requirement | Suitable approach |
|---|---|
| Read-only API that reflects later owner changes | `MappingProxyType(backing_dict)` |
| Independent, mutable top-level snapshot | `dict(proxy)` or `proxy.copy()` |
| Independent nested snapshot | `deepcopy(dict(proxy))`, when appropriate |
| Deeply read-only common containers | recursive freezing strategy |
| Function only needs reads | type as `Mapping[K, V]` |
| Function intentionally mutates mapping | type as `MutableMapping[K, V]` |
| Existing proxy must survive refresh | mutate same backing dict |
| Concurrent multi-step invariants | synchronization / concurrency design |